# 02A - Tokenization and Embeddings Theory

**Guiding question:** how do you turn any scribbled pet-sitter's note into compact, trainable addresses without letting padding become fake evidence?

> **What you already know:** 01B consumes integer token IDs, shifted targets, and real-token masks.
> **The gap this chapter closes:** where those IDs come from, what embedding lookup can and cannot represent, and why masking changes the learning objective.
> **What you'll have by the end:** measured BPE compression, a token-address mental model, and a visible padding failure.
> **What this chapter is not:** a tokenizer benchmark or broad contextual-embedding survey.

We'll stay in a cosy living room with a cat and a dog the whole way: pet-sitter's notes,
a `sleepy` cat on the mat, and the word `bark` with two meanings to untangle later.


In [ ]:
# -- Setup and running pet corpus ---------------------------------------------
from collections import Counter
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

PET_CORPUS = [
    "the sleepy cat sat on the mat",
    "the dog sat on the rug",
    "the cat slept on the rug",
    "a sleepy dog slept on the mat",
    "the loud bark woke the cat",
    "the rough bark covered the tree",
]
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})
print(f"Pet examples: {len(PET_CORPUS)}")


## 1 - Character coverage or word efficiency?

![The new word sleepyhead split three ways: ten single letters, one unknown whole word, or the learned piece sleepy plus the letters h, e, a, d.](images/02-01-three-ways-to-split.png)

*Letters never get stuck but run long; whole words are short but choke on anything new. BPE sits in between.*

**Predict:** which vocabulary is larger on this corpus: unique characters or unique words? Which one creates the longer sequence?


In [ ]:
# -- Measure the two crude tokenization extremes ---------------------------------
text = " ".join(PET_CORPUS)
characters = list(text)
words = re.findall(r"[a-z-]+", text.lower())
print(f"Character vocabulary: {len(set(characters))}; sequence length: {len(characters)}")
print(f"Word vocabulary:      {len(set(words))}; sequence length: {len(words)}")
print("Takeaway: characters guarantee coverage; words spend fewer positions but break on novelty.")


Characters cover unfamiliar spellings but make long sequences; whole words are compact but fail on new spellings. A small rule can repeatedly join neighboring pieces that often appear together. That rule is **byte-pair encoding (BPE)**, and this example uses it to balance coverage with shorter sequences.

## 2 - BPE closes the most common seam

The first attempt is deliberately small: words become character tuples with an end marker. Every merge changes only one adjacent pair.

![BPE loop: count neighbouring pairs, glue the most common pair, repeat. One learned piece, at, is reused by cat, sat and mat.](images/02-02-bpe-glues-common-pairs.png)

*Each pass glues one common pair. The glued piece then gets reused wherever it shows up.*


In [ ]:
# -- Compact BPE mechanics: count, merge, record ---------------------------------
def initialize_words(corpus):
    counts = Counter(re.findall(r"[a-z-]+", " ".join(corpus).lower()))
    return {tuple(word) + ("</w>",): frequency for word, frequency in counts.items()}

def pair_counts(vocabulary):
    counts = Counter()
    for symbols, frequency in vocabulary.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += frequency
    return counts

def merge_pair(vocabulary, pair):
    merged = {}
    for symbols, frequency in vocabulary.items():
        output = []
        index = 0
        while index < len(symbols):
            if index + 1 < len(symbols) and (symbols[index], symbols[index + 1]) == pair:
                output.append(symbols[index] + symbols[index + 1])
                index += 2
            else:
                output.append(symbols[index])
                index += 1
        merged[tuple(output)] = frequency
    return merged

vocabulary = initialize_words(PET_CORPUS)
target_word = tuple("sleepy") + ("</w>",)
merge_history = []
target_history = [target_word]

for _ in range(18):
    counts = pair_counts(vocabulary)
    if not counts:
        break
    best_pair, frequency = counts.most_common(1)[0]
    vocabulary = merge_pair(vocabulary, best_pair)
    target_word = next(
        symbols for symbols in vocabulary
        if "".join(symbols).replace("</w>", "") == "sleepy"
    )
    merge_history.append((best_pair, frequency))
    target_history.append(target_word)

print(f"Initial target pieces: {len(target_history[0])}")
print(f"After {len(merge_history)} merges: {len(target_history[-1])}")
print(f"Final pieces: {target_history[-1]}")


In [ ]:
# -- Static BPE storyboard: essential evidence without JavaScript ----------------
selected_steps = sorted(set([0, 1, 4, 8, len(target_history) - 1]))
counts = [len(target_history[index]) for index in selected_steps]
labels = [f"merge {index}" for index in selected_steps]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, counts, color="#60a5fa")
ax.set(title="'sleepy' compression", ylabel="Piece count")
for bar, index in zip(bars, selected_steps):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        " | ".join(target_history[index]),
        ha="center",
        va="bottom",
        rotation=15,
        color="white",
        fontsize=8,
    )
plt.show()


In [ ]:
# -- BPE animation: every frame comes from recorded merge state ------------------
piece_counts = [len(state) for state in target_history]
fig, ax = plt.subplots(figsize=(9, 4))
ax.set(xlim=(-0.5, len(piece_counts) - 0.5), ylim=(0, max(piece_counts) + 2))
ax.set(xlabel="Merge step", ylabel="Pieces in 'sleepy'")
line, = ax.plot([], [], color="#34d399", linewidth=3, marker="o")

def update_bpe(frame):
    visible = frame + 1
    line.set_data(range(visible), piece_counts[:visible])
    ax.set_title(" | ".join(target_history[frame]), color="white")
    return (line,)

bpe_animation = FuncAnimation(fig, update_bpe, frames=len(piece_counts), interval=400, blit=True)
plt.close(fig)
print("Watch common seams close while complete character coverage remains available.")
display(HTML(bpe_animation.to_jshtml()))


### That compressed the string. It still produced symbols, not model inputs.

For `sleepy`, the run above shrank 7 initial symbols (including the end marker) to 1 piece after 18 merges.

A vocabulary turns each piece into an address. The address chooses one row; numeric size has no semantic meaning.


Once the text is split into pieces, give each piece a stable integer address. The number is for looking up the piece, not for measuring its meaning.

### Where do the numbers in that row come from?

An ID is like a name tag. It tells us which word we have, not what might help us use it.
An embedding row is a bundle of numbers we can feed into a network. But merely having
numbers doesn't mean we have useful features.

In the next example, we make those numbers **randomly** with `rng.normal`. Nothing has
trained them, and we haven't loaded someone else's learned word vectors. The example checks
one small fact: `bark` fetches the same stored row in two sentences, while mixing in a
different neighbour changes the result. That hand-written mixing rule is not learned
context understanding. Don't read the random numbers as word meaning.

Why train a table later? Think about `cat` and `dog`: in our examples, both
can be followed by `sat` or `slept`. A predictor could learn separate rules for two
arbitrary random rows. Or we can let training adjust the rows so the same prediction layers
can use features helpful for both words. We are learning the inputs as well as the predictor.
The useful features need not be human-labelled columns such as "furriness".

**Here is the payoff: different words can share useful features instead of being unrelated
name tags.** If training teaches the model something about `cat`, those shared features
can help it handle `dog` in a similar setting. This is one reason learned embeddings
matter: they can capture patterns of meaning and use that arbitrary IDs don't express.
It isn't just changing numbers; it is giving the rest of the network a way to reuse what
it learned across words. Whether that reuse works on a new example still needs testing.

Don't read "similar" as "identical". The words keep different IDs and rows. Words used in
similar contexts can develop similar features, but that can include opposites such as
`happy` and `sad` in "the dog felt ___". Nearby vectors suggest a learned relationship, not a guarantee
that one word can replace the other. And one static row can't choose which meaning of
`bark` a sentence intends; that needs the surrounding words too.

Random rows are not unusable: a large enough network can learn from fixed random identifiers,
and might memorize a tiny dataset. Learned rows give the model another way to organize what
it knows, rather than forcing the later layers to do all the work. Whether that helps on
new examples still needs testing. Neither low training loss nor a pretty embedding plot
proves that we've learned general meaning.

02B will start a PyTorch table at random and train it on a few supplied word pairs.
Chapter 03 then makes those practice questions from text instead of handing them over.

## 3 - Token IDs address static rows

![The piece ' cat' gets ID 3797, which opens drawer 3797 in GPT-2's table of 50,257 drawers and fetches one row of learnable numbers. Both meanings of bark open the same drawer.](images/02-03-id-opens-a-drawer.png)

*The ID is just a drawer number. Both meanings of bark open the same drawer; only the sentence can tell them apart.*

**Predict:** before context mixing, should `bark` begin with different lookup vectors in `loud bark` and `rough bark`?


In [ ]:
# -- Static lookup versus contextualized token representation --------------------
rng = np.random.default_rng(42)
small_vocab = ["loud", "bark", "rough", "tree", "woke"]
token_to_id = {token: index for index, token in enumerate(small_vocab)}
table = rng.normal(size=(len(small_vocab), 4))

# An address selects [V,4] -> [4]; both bark occurrences fetch the very same stored row.
bark_lookup_loud = table[token_to_id["bark"]]
bark_lookup_rough = table[token_to_id["bark"]]
# Mix two [4] rows element by element; the scalar 0.5 broadcasts across features, shape stays [4].
loud_context = bark_lookup_loud + 0.5 * table[token_to_id["loud"]]
rough_context = bark_lookup_rough + 0.5 * table[token_to_id["rough"]]

print(f"Static bark rows equal: {np.array_equal(bark_lookup_loud, bark_lookup_rough)}")
print(f"After neighbor mixing, contextual rows equal: {np.array_equal(loud_context, rough_context)}")
print("Boundary: tokenization supplies addresses; a later model performs context mixing.")


A batch needs rows of equal width, so shorter examples get empty `<pad>` slots. Those slots are bookkeeping, not evidence from the pet-sitter's notes, and they should not count as correct prediction targets.

## 4 - Padding creates fake positions unless the objective refuses them

![A two-row batch, 'the cat sat' and 'the dog', padded to five seats. Averaging every cell lets blanks water down the score; averaging only real words keeps it honest.](images/02-04-padding-is-not-evidence.png)

*Blank seats keep the batch rectangular. They must not count towards the score.*


In [ ]:
# -- Visible padding failure: one rectangle, two denominators --------------------
token_losses = np.array([
    [0.20, 0.40, 0.30, 0.05, 0.05],
    [0.25, 0.35, 0.05, 0.05, 0.05],
])
real_mask = np.array([
    [1, 1, 1, 0, 0],
    [1, 1, 0, 0, 0],
], dtype=bool)
# [2,5] -> scalar includes blanks; Boolean indexing below first keeps only the five real losses [5].
wrong_mean = token_losses.mean()
correct_mean = token_losses[real_mask].mean()

fig, ax = plt.subplots(figsize=(8, 3))
image = ax.imshow(real_mask, cmap="Blues", vmin=0, vmax=1)
ax.set(xticks=range(5), yticks=range(2), xlabel="Batch position", ylabel="Sequence")
ax.set_title("Real-token mask: blue is evidence, dark is padding")
plt.colorbar(image, ax=ax)
plt.show()

print(f"Wrong all-cell mean: {wrong_mean:.3f}")
print(f"Correct real-token mean: {correct_mean:.3f}")
print("Takeaway: padding changes the objective unless the denominator excludes it.")


## 5 - Production bridge and handoff

GPT-2 uses byte-level BPE: bytes guarantee coverage, learned merges shorten common sequences, and whitespace can be part of a token piece. The lab inspects those real IDs with `tiktoken`.

| Teaching mechanism | Production counterpart | What changes | What does not |
|---|---|---|---|
| character-tuple BPE | byte-level BPE | base symbols and merge vocabulary | frequent pairs become reusable pieces |
| tiny ID map | GPT-2 vocabulary | scale and pretrained artifact | IDs remain addresses |
| static NumPy table | `torch.nn.Embedding` | trainable framework storage | lookup selects rows |
| Boolean real-token mask | `ignore_index` and attention masks | API and mask shape | padding is not evidence |

**Built and measured:** BPE compression, static lookup boundary, padding-loss failure.

**Explained or illustrated:** production byte-level BPE and contextualization boundary.

**Named and out of scope:** unigram tokenizers, multilingual vocabulary design, and causal attention masks.

Next, [02B](02B-tokenization-and-embeddings-pytorch-lab.ipynb) turns every contract into PyTorch code.
